In [3]:
import os
import sys
import time
import requests
import argparse
import json
from requests.exceptions import RequestException

In [4]:
api_key = "7eDRxpakYIVqUdC0y-S2a3WZrISixZ0pFVT1DiUMIaCB"
project_id = "69feff15-20d1-4895-a4e8-bf660de13587"
url = "https://eu-de.ml.cloud.ibm.com/ml/v1/text/generation?version=2023-05-29"
max_new_tokens = 900


class IBMWatsonXAIWrapper:
    def __init__(self, api_key, project_id, url, model_id="sdaia/allam-1-13b-instruct", max_new_tokens=400, decoding_method="greedy", temperature=0.7, top_p=1, repetition_penalty=1.0, timeout=60):
        self.api_key = api_key
        self.project_id = project_id
        self.base_url = url
        self.url = f"{url}/ml/v1/text/generation?version=2023-05-29"
        self.model_id = model_id
        self.timeout = timeout
        
        self.parameters = {
            "decoding_method": decoding_method,
            "max_new_tokens": max_new_tokens,
            "temperature": temperature,
            "top_p": top_p,
            "repetition_penalty": repetition_penalty
        }
        
        self.access_token = self.get_access_token()
        self.headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.access_token}"
        }
        
        print(f"Debug: Initialized with model_id: {self.model_id}")

    def get_access_token(self):
        """
        Obtain an access token from IBM Cloud IAM.
        This method sends a POST request to the IBM Cloud IAM token endpoint to
        retrieve an access token using the provided API key.
        Returns:
            str: The access token.
        Raises:
            RequestException: If there is an error obtaining the access token.
            SystemExit: If the request fails, the program will exit with status 1.
        """
        token_url = "https://iam.cloud.ibm.com/identity/token"
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        data = {
            "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
            "apikey": self.api_key
        }
        
        try:
            response = requests.post(token_url, headers=headers, data=data, timeout=self.timeout)
            response.raise_for_status()
            return response.json()["access_token"]
        except RequestException as e:
            print(f"Error obtaining access token: {str(e)}")
            sys.exit(1)

    

    def generate_text(self, prompt):
        """
        Generates text based on the provided prompt using the specified model.
        Args:
            prompt (str): The input text to generate a response for.
        Returns:
            str: The generated text from the model, or an error message if the request fails.
        Raises:
            Exception: If the API response status code is not 200.
        """
        
        body = {
            "input": f"<s> [INST] {prompt} [/INST]",
            "parameters": self.parameters,
            "model_id": self.model_id,
            "project_id": self.project_id
        }
        
        try:
            print("Generating response...", end="", flush=True)
            response = requests.post(
                self.url,
                headers=self.headers,
                json=body,
                timeout=self.timeout
            )
            print("\rGeneration complete.   ")
            
            if response.status_code != 200:
                raise Exception(f"Non-200 response: {response.text}")
            
            data = response.json()
            return data.get('results', [{}])[0].get('generated_text', "No text generated")
        except RequestException as e:
            print(f"\nError: API request failed - {str(e)}")
            return f"Error: API request failed - {str(e)}"
        except Exception as e:
            print(f"\nError: {str(e)}")
            return f"Error: {str(e)}"


In [11]:
if __name__ == "__main__":
    try:
        wrapper = IBMWatsonXAIWrapper(
            api_key=api_key,
            project_id=project_id,
            url=url,
            max_new_tokens=max_new_tokens,
            decoding_method="greedy",
            temperature=0.7,
            top_p=1,
            repetition_penalty=1.9,
            timeout=60
        )
            
        user_input = "please tell me about love in Arabic"            
        response = wrapper.generate_text(user_input)
        
        print(f"ALLaM: {response}")
    except Exception as e:
        
        print(f"Error: {str(e)}")
        print("Please check your API key, project ID, and URL, and make sure you have the correct permissions.")

Debug: Initialized with model_id: sdaia/allam-1-13b-instruct
Generation complete.   
ALLaM:  فِي الْعَرَبِيَّةِ، يُطْلَق عَلَى الْحُبِّ اسْم "الْوُدّ" أَوْ "العِشْق". الحّب يعتبر جزءاً أساسياً من الثقافة العربية والإسلامية، حيث يتجسد في العلاقات الأسرية والاجتماعية والرومانسية.

1. الود: الوِد هو الحب العميق والصداقة القوية التي تجمع بين الأشخاص بناءً على الاحترام والتقدير المتبادل والتفاهم الجيد بينهم وبين بعضهم البعض وفي إطار الأسرة والمجتمع بشكل عام. يمكن أن يكون هذا النوع بسيطاً ومستمراً لفترات طويلة دون تقلبات عاطفية شديدة أو رومانسية مفرطة كما يحدث غالباً مع العشق الرومانسي التقليدي الغربي (Romantic Love).
2. العشق والغرام العربي الإسلامي قد يختلف عن المفهوم الشائع للعلاقة الرومانسية الغربية؛ إذ يميل إلى التركيز أكثر حول القيم والأخلاق والاحترام المشترك بدلاً مِن التعلق الشديد بالمشاعر العاطفة والجسدية فقط والتي تعتبر أقل ظهوراً مقارنة بالثقافة الأوروبية والأمريكية مثلاً. ومع ذلك يظل مفهوم الغرام والعواطف العميقة موجوداً ضمن السياق الثقافي والديني للمجتمع المسلم والعربي عموماً لك